# Read counts after a pasilla knockdown

The first cell holds the data: seven sample names and the read counts of six genes.
Run it before anything else, and again after every kernel restart.

In [ ]:
# RNA-seq read counts of six genes in seven samples of Drosophila melanogaster
# Source: the Bioconductor data package pasilla (see the experiment's README)

# The sample names, in the same order as the counts below
samples = ["ctrl_rep1", "ctrl_rep2", "ctrl_rep3", "ctrl_rep4", "kd_rep1", "kd_rep2", "kd_rep3"]

# Read counts: one list per gene (FlyBase gene ID), one number per sample
counts = {
    "FBgn0000008": [92, 161, 76, 70, 140, 88, 70],
    "FBgn0000014": [5, 1, 0, 0, 4, 0, 0],
    "FBgn0000042": [101664, 120163, 45880, 53201, 127363, 76099, 83164],
    "FBgn0025111": [451, 653, 279, 306, 4186, 2381, 2686],
    "FBgn0039155": [1527, 2220, 741, 931, 70, 38, 52],
    "FBgn0261552": [7022, 16750, 5224, 4760, 4233, 1335, 1394],
}

## EX1: Python syntax (15 min, on your own)

Genes with very few reads tell you little, so you want to know which genes have enough.
An LLM writes the code; to judge it, you first look at how Python code reads.

1. Run the cell `# Variables and data types`. Two genes have fewer reads than 1000.
2. Copy the first code cell and an instruction to solve `# your task` into the LLM chat,
   for example:

   > Using the dictionary `counts` above, write plain Python that prints the genes whose
   > mean count over all seven samples is above `threshold = 100`.

3. Paste the code into the `# your task` cell and run it. Which line is the condition? If
   it fails, paste the error message into the chat.
4. Save the notebook (Ctrl + S) and open the Git panel (Git icon in the left sidebar). It
   should list `explore_counts.ipynb`. Write a short message, for example
   `list genes above threshold`, and commit.

In [ ]:
# Variables and data types
gene = "FBgn0000008"            # a text (str)
threshold = 1000                # a whole number (int)
first_count = counts[gene][0]   # look up the gene in the dictionary, then the first item in its list
print(gene, type(gene), threshold, type(threshold), first_count)

# A loop over the dictionary, with a condition inside it
for gene_id, values in counts.items():
    total = sum(values)
    if total < threshold:
        print(gene_id, "has fewer reads:", total, "than threshold:", threshold)
    else:
        print(gene_id, "has", total, "reads in", len(values), "samples")

In [ ]:
# your task: find the genes whose mean count over all samples is above the threshold of 100


## EX2: Debugging (20 min, in groups)

You want to know the mean count over each set of replicates. Sample names look like
`ctrl_rep1`: a condition, an underscore, `rep` and a replicate number. To calculate
the mean you want to group the sets and for that you want to parse the sample names.

1. Read the parser cell together before you run it. One of you explains what each line
   should do.
2. Run it. You should see `ValueError: invalid literal for int() with base 10: 'rep1'`.
   Read the traceback from the bottom up: which line failed, and which line called it?
3. Paste

   - the data cell from above
   - the parser cell
   - the error message

   into the chat and ask for a fix. Read the fix, run it, then run `# check the result`.
   You should see seven names such as `'ctrl_rep1': {'condition': 'ctrl', 'replicate': 1}`.
4. The cell `# mean count per condition` computes the mean over each set of replicates,
   but its steps have no comments. Paste it into the chat and ask the LLM to add a comment
   to each step. Read the comments, paste the code back into the cell and run it. You
   should see a mean for `ctrl` and `kd` per gene, such as
   `FBgn0039155 {'ctrl': 1354.8, 'kd': 53.3}`.
5. Save the notebook and open the Git panel. It should list `explore_counts.ipynb`. Write
   a short message, for example `fix sample name parser`, and commit.

In [ ]:
def parse_sample_name(name):
    """Split a sample name like 'ctrl_rep1' into its condition and replicate number."""
    condition, replicate = name.split("_")
    return {"condition": condition, "replicate": int(replicate)}

# Parse all sample names
sample_info = {}
for name in samples:
    sample_info[name] = parse_sample_name(name)

In [ ]:
# check the result
from pprint import pprint
pprint(sample_info)

In [ ]:
# mean count per condition
positions = {}
for i, name in enumerate(samples):
    condition = sample_info[name]["condition"]
    if condition not in positions:
        positions[condition] = []
    positions[condition].append(i)

for gene_id, values in counts.items():
    means = {}
    for condition, indices in positions.items():
        group = [values[i] for i in indices]
        means[condition] = round(sum(group) / len(group), 1)
    print(gene_id, means)

## EX3: Improvement (30 min, homework)

New samples are named by hand, and some names will break the naming rule. Make the parser
more reliable, then complete the experiment entry with what you found.

1. The rule is `<condition>_rep<number>`, where the condition is `ctrl` or `kd`. Ask the
   LLM to change your fixed `parse_sample_name` so that it reports each malformed name and
   why, and goes on with the next one. Give it the rule.
2. Run it on `samples + more_samples`. Check by hand which names break the rule: does the
   code report exactly those?
3. In the experiment's `README.md`, answer the questions under "Observations and
   interpretation". In `metadata.yml`, fill in `date_started`, `author` and `status` (its
   comment lists the values).
4. Save the notebook and open the Git panel. It should list `explore_counts.ipynb`,
   `README.md` and `metadata.yml`. Write a short message, for example
   `report malformed sample names`, and commit.

In [ ]:
# Made-up sample names to test the parser.
more_samples = ["ctrl_rep5", "kd-rep4", "kd_rep", "KD_rep5", "kd_rep4", "ctrl_rep6_new", "ctrl8"]